# Notebook 02: Análise de Dados de Saúde — DataSUS

**Projeto:** 
**Objetivo:** Analisar dados reais do SIM (óbitos) e SINASC (nascimentos) para a Região Oeste de SC (2018–2023).

Municípios da Região Oeste: Chapecó, Concórdia, Seara, São Miguel do Oeste, Itapiranga, Guatambú, Xanxerê, Joaçaba, Herval d'Oeste, Xaxim, Cunha Porã, Cunhataí, Guaraciaba, Jupiá, Riqueza, Vargeão, Águas de Chapecó, Bom Jesus do Oeste, Coronel Freitas, Caxambu do Sul.


## 1. Configuração do Ambiente

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

# Adiciona src ao path
PROJECT_ROOT = Path(os.getcwd()).resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from config import SETTINGS

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

DATA_PROCESSED = SETTINGS.DATA_PROCESSED
print("Dados processados:", DATA_PROCESSED)


## 2. Leitura dos Dados Agregados

In [ ]:
# Leitura dos Parquets gerados pelo pipeline
df_sim = pl.read_parquet(DATA_PROCESSED / "datasus/sim_agregado_oeste_sc.parquet")
df_sin = pl.read_parquet(DATA_PROCESSED / "datasus/sinasc_agregado_oeste_sc.parquet")
df_ind = pl.read_parquet(DATA_PROCESSED / "indicadores_demograficos_oeste_sc.parquet")

# Conversão para pandas (facilita plotagem)
sim = df_sim.to_pandas()
sin = df_sin.to_pandas()
ind = df_ind.to_pandas()

print("SIM registros:", len(sim), "| Anos:", sorted(sim.ano.unique()))
print("SINASC registros:", len(sin), "| Anos:", sorted(sin.ano.unique()))
print("Indicadores registros:", len(ind), "| Anos:", sorted(ind.ano.unique()))
display(sim.head(3))


## 3. Série Temporal — Óbitos Totais por Município

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
for municipio, grp in sim.groupby("municipio"):
    grp = grp.sort_values("ano")
    ax.plot(grp["ano"], grp["total_obitos"], marker="o", label=municipio, alpha=0.8)
ax.set_title("Óbitos Totais por Município — Região Oeste de SC (2018–2023)")
ax.set_xlabel("Ano")
ax.set_ylabel("Total de Óbitos")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


## 4. Série Temporal — Nascimentos Totais por Município

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
for municipio, grp in sin.groupby("municipio"):
    grp = grp.sort_values("ano")
    ax.plot(grp["ano"], grp["total_nascimentos"], marker="o", label=municipio, alpha=0.8)
ax.set_title("Nascimentos Totais por Município — Região Oeste de SC (2018–2022)")
ax.set_xlabel("Ano")
ax.set_ylabel("Total de Nascimentos")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


## 5. Comparativo — Chapecó vs Demais Municípios

In [ ]:
sim["grupo"] = sim["municipio"].apply(lambda x: "Chapecó" if x == "Chapecó" else "Demais")
sin["grupo"] = sin["municipio"].apply(lambda x: "Chapecó" if x == "Chapecó" else "Demais")

agg_sim = sim.groupby(["ano", "grupo"])["total_obitos"].sum().reset_index()
agg_sin = sin.groupby(["ano", "grupo"])["total_nascimentos"].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for grupo, grp in agg_sim.groupby("grupo"):
    grp = grp.sort_values("ano")
    axes[0].plot(grp["ano"], grp["total_obitos"], marker="o", label=grupo, linewidth=2.5)
axes[0].set_title("Óbitos: Chapecó vs Demais")
axes[0].set_xlabel("Ano")
axes[0].set_ylabel("Total de Óbitos")
axes[0].legend()

for grupo, grp in agg_sin.groupby("grupo"):
    grp = grp.sort_values("ano")
    axes[1].plot(grp["ano"], grp["total_nascimentos"], marker="o", label=grupo, linewidth=2.5)
axes[1].set_title("Nascimentos: Chapecó vs Demais")
axes[1].set_xlabel("Ano")
axes[1].set_ylabel("Total de Nascimentos")
axes[1].legend()

plt.tight_layout()
plt.show()


## 6. Indicadores Demográficos — Taxas Brutas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.lineplot(data=ind, x="ano", y="taxa_bruta_mortalidade", hue="municipio", marker="o", ax=axes[0], legend=False, alpha=0.7)
axes[0].set_title("Taxa Bruta de Mortalidade (por 1.000 hab.)")
axes[0].set_xlabel("Ano")
axes[0].set_ylabel("‰")

# Remove nulls para natalidade (2023 sem SINASC)
ind_natal = ind.dropna(subset=["taxa_bruta_natalidade"])
sns.lineplot(data=ind_natal, x="ano", y="taxa_bruta_natalidade", hue="municipio", marker="o", ax=axes[1], legend=False, alpha=0.7)
axes[1].set_title("Taxa Bruta de Natalidade (por 1.000 hab.)")
axes[1].set_xlabel("Ano")
axes[1].set_ylabel("‰")

plt.tight_layout()
plt.show()


## 7. Taxa de Mortalidade Infantil (< 1 ano)

In [ ]:
ind_mi = ind.dropna(subset=["taxa_mortalidade_infantil"])
fig, ax = plt.subplots(figsize=(14, 7))
for municipio, grp in ind_mi.groupby("municipio"):
    grp = grp.sort_values("ano")
    ax.plot(grp["ano"], grp["taxa_mortalidade_infantil"], marker="o", label=municipio, alpha=0.8)
ax.set_title("Taxa de Mortalidade Infantil por Município (2018–2022)")
ax.set_xlabel("Ano")
ax.set_ylabel("Óbitos < 1 ano / 1.000 nascidos vivos")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


## 8. Distribuição de Óbitos por Sexo e Faixa Etária

In [ ]:
faixa_cols = [c for c in sim.columns if c.startswith("obitos_faixa_")]
sim_faixa = sim[["municipio", "ano"] + faixa_cols].copy()
sim_faixa = sim_faixa.melt(id_vars=["municipio", "ano"], value_vars=faixa_cols, var_name="faixa_etaria", value_name="obitos")
sim_faixa["faixa_etaria"] = sim_faixa["faixa_etaria"].str.replace("obitos_faixa_", "")

fig, ax = plt.subplots(figsize=(14, 7))
ordem = ["0-1", "1-4", "5-14", "15-29", "30-39", "40-59", "60+"]
sns.boxplot(data=sim_faixa, x="faixa_etaria", y="obitos", order=ordem, ax=ax)
ax.set_title("Distribuição de Óbitos por Faixa Etária — Região Oeste de SC")
ax.set_xlabel("Faixa Etária")
ax.set_ylabel("Óbitos por município-ano")
plt.tight_layout()
plt.show()


## 9. Nascimentos — Peso ao Nascer e Mães Adolescentes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=sin, x="ano", y="nascimentos_peso_baixo", ax=axes[0])
axes[0].set_title("Nascimentos com Peso < 2500g por Ano")
axes[0].set_xlabel("Ano")
axes[0].set_ylabel("Quantidade")

sns.boxplot(data=sin, x="ano", y="nascimentos_mae_adolescente", ax=axes[1])
axes[1].set_title("Nascimentos com Mãe Adolescente (< 20 anos) por Ano")
axes[1].set_xlabel("Ano")
axes[1].set_ylabel("Quantidade")

plt.tight_layout()
plt.show()


## 10. Tipos de Parto (Cesárea vs Normal)

In [ ]:
sin_parto = sin[["municipio", "ano", "partos_cesarea", "partos_normal"]].copy()
sin_parto = sin_parto.melt(id_vars=["municipio", "ano"], value_vars=["partos_cesarea", "partos_normal"], var_name="tipo_parto", value_name="quantidade")
sin_parto["tipo_parto"] = sin_parto["tipo_parto"].map({"partos_cesarea": "Cesárea", "partos_normal": "Normal"})

fig, ax = plt.subplots(figsize=(14, 7))
sns.barplot(data=sin_parto, x="ano", y="quantidade", hue="tipo_parto", ax=ax, estimator="sum", errorbar=None)
ax.set_title("Total de Partos por Tipo — Região Oeste de SC")
ax.set_xlabel("Ano")
ax.set_ylabel("Quantidade")
plt.tight_layout()
plt.show()


## 11. Principais Causas de Óbito (Top CID-10)

In [ ]:
from collections import Counter

causas = []
for _, row in sim.iterrows():
    for causa, n in zip(row["top3_causas"], row["top3_causas_n"]):
        causas.append((causa, n))

contagem = Counter()
for causa, n in causas:
    contagem[causa] += n

top10 = contagem.most_common(10)
causas_df = pd.DataFrame(top10, columns=["capitulo_cid10", "obitos"])

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=causas_df, x="obitos", y="capitulo_cid10", palette="viridis", ax=ax)
ax.set_title("Top 10 Capítulos CID-10 — Óbitos na Região Oeste de SC (2018–2023)")
ax.set_xlabel("Total de Óbitos (soma top-3 por município-ano)")
ax.set_ylabel("Capítulo CID-10")
plt.tight_layout()
plt.show()


## 12. Resumo por Município (Média Anual)

In [ ]:
resumo = ind.groupby("municipio").agg({
    "total_obitos": "mean",
    "total_nascimentos": "mean",
    "populacao": "mean",
    "taxa_bruta_mortalidade": "mean",
    "taxa_bruta_natalidade": "mean",
    "taxa_mortalidade_infantil": "mean"
}).round(2)
resumo = resumo.sort_values("total_obitos", ascending=False)
display(resumo)


## Próximos Passos

- Cruzar dados de mortalidade/natalidade com indicadores socioeconômicos e migratórios.
- Analisar evolução temporal das principais causas de óbito.
- Incorporar dados de população estrangeira/venezuelana quando disponíveis.
